In [17]:
QUERY = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX wikibase: <http://wikibase.xyz/ontology#>
PREFIX bd: <http://www.bigdata.com/rdf#>
PREFIX mwapi: <https://www.mediawiki.org/ontology/canonical/api#>
PREFIX schema: <http://schema.org/>

SELECT DISTINCT ?person ?personLabel ?article
WHERE {
  # 1. 安全なWikidata自身のエンドポイントから、日本語版Wikipedia(jawiki)のカテゴリメンバーを直接スキャンする
  SERVICE wikibase:mwapi {
    bd:serviceParam wikibase:endpoint "www.wikidata.org" ;
                    wikibase:api "Generator" ;
                    mwapi:generator "categorymembers" ;
                    # 対象カテゴリ。日本語版Wikipediaの存命人物は、Wikidata上で「jawiki:Category:存命人物」と指定
                    mwapi:gcmtitle "jawiki:Category:存命人物" ; 
                    mwapi:gcmlimit "max" .
    
    # カテゴリから対応するWikidataのQID（?person）をダイレクトに取り出す
    ?person wikibase:apiOutputItem mwapi:item .
  }

  # 2. 対象が人間(Q5)かつ日本国籍(Q17)であることを担保
  ?person wdt:P31 wd:Q5 ;
          wdt:P27 wd:Q17 .

  # 3. 日本語WikipediaのURLも合わせてマッピング
  OPTIONAL {
    ?article schema:about ?person ;
             schema:isPartOf <https://ja.wikipedia.org/> .
  }

  # 4. 日本語ラベルの取得
  SERVICE wikibase:label { bd:serviceParam wikibase:language "ja". }
}
"""
HEADERS = {
    "User-Agent": "MyBot/1.0 (contact: your_email@example.com)",
    "Accept": "application/sparql-results+json",
}

In [18]:
import os
import json
import time
import requests
import pandas as pd

os.makedirs("data", exist_ok=True)

QLEVER_URL = "https://qlever.cs.uni-freiburg.de/api/wikidata"

JSON_PATH = "data/test.json"
CSV_PATH = "data/test.csv"


def fetch_json(max_retries=5):
    """SPARQL標準のJSON結果形式で取得する。action指定なし = デフォルトのJSON。"""
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Sending request to QLever (Attempt {attempt}/{max_retries})...")
            r = requests.post(
                QLEVER_URL,
                data={"query": QUERY},  # action は付けない = SPARQL標準JSON
                headers=HEADERS,
                timeout=300,
            )
            if r.status_code == 200:
                return r.json()
            else:
                print(f"  Status error: {r.status_code}, body[:300]={r.text[:300]!r}")
        except requests.exceptions.RequestException as e:
            print(f"  Network error: {e}")
        except json.JSONDecodeError as e:
            print(f"  JSON decode error: {e}, body[:300]={r.text[:300]!r}")

        time.sleep(5 * attempt)

    raise RuntimeError(f"Failed to fetch data after {max_retries} retries.")


def json_to_dataframe(data: dict) -> pd.DataFrame:
    """SPARQL JSON results (head.vars / results.bindings) を DataFrame化する。"""
    cols = data["head"]["vars"]
    bindings = data["results"]["bindings"]

    rows = []
    for b in bindings:
        row = {col: b.get(col, {}).get("value") for col in cols}
        rows.append(row)

    return pd.DataFrame(rows, columns=cols)


def main():
    start_time = time.time()

    # 1. JSON取得
    data = fetch_json(10)
    n_bindings = len(data.get("results", {}).get("bindings", []))
    print(f"Fetched {n_bindings} rows from QLever.")

    # 2. JSON保存（生データをそのまま残す。デバッグ・再変換用）
    with open(JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"Saved raw JSON -> {JSON_PATH}")

    # 3. CSV化して保存
    df = json_to_dataframe(data)
    df.to_csv(CSV_PATH, index=False, encoding="utf-8")
    print(f"Saved CSV -> {CSV_PATH}")

    elapsed = time.time() - start_time
    print(f"\nDone. Rows: {len(df)}, Columns: {list(df.columns)}")
    print(f"Total time spent: {int(elapsed)}s")


if __name__ == "__main__":
    main()

Sending request to QLever (Attempt 1/10)...
  Status error: 500, body[:300]='{\n    "exception": "Error while executing a SERVICE request to <http://wikibase.xyz:80/ontology#mwapi>: SERVICE responded with HTTP status code: 400, Bad Request. First 100 bytes of the response: \'<html><body><h1>400 Bad request</h1>\\nYour browser sent an invalid request.\\n</body></html>\\n\'",\n    "q'
Sending request to QLever (Attempt 2/10)...
  Status error: 500, body[:300]='{\n    "exception": "Error while executing a SERVICE request to <http://wikibase.xyz:80/ontology#mwapi>: SERVICE responded with HTTP status code: 400, Bad Request. First 100 bytes of the response: \'<html><body><h1>400 Bad request</h1>\\nYour browser sent an invalid request.\\n</body></html>\\n\'",\n    "q'
Sending request to QLever (Attempt 3/10)...
  Status error: 500, body[:300]='{\n    "exception": "Error while executing a SERVICE request to <http://wikibase.xyz:80/ontology#mwapi>: SERVICE responded with HTTP status code: 400, 

KeyboardInterrupt: 

In [9]:
test = pd.read_csv("data/test.csv")

In [10]:
test.shape

(0, 3)

In [3]:
1 * 34881 / 1000

34.881